# Finding Z: editable hypothesis analysis

Choose signal and background/null hypotheses, explore distributions, define a signal region, and run a cut-and-count. Instructor-provided and student-generated samples use the same saved-run format and normalization.

In [ ]:
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from findingz.catalog import load_catalog
from findingz.counting import summarize_cut_and_count, weighted_yield
from findingz.delphes import default_run_root, open_run
from findingz.hypotheses import (
    build_sample_library, select_events, validate_analysis_context,
    validate_counting_samples,
)

## 1. Inspect and choose hypotheses

Copy any `sample_id` from this table into the editable choices below. Generated runs retain their run names, configurations, event counts, and MadGraph cross sections.

In [ ]:
library = build_sample_library(load_catalog(), default_run_root())
pd.DataFrame([
    {
        'sample_id': sample.sample_id,
        'label': sample.label,
        'generated_events': sample.generated_events,
        'cross_section_pb': sample.cross_section_pb,
        'configuration': sample.config,
    }
    for sample in library.values()
])

In [ ]:
# Choose compatible run:<run_id> values from the table; the roles are your choice.
sample_ids = list(library)
assert len(sample_ids) >= 2, 'Generate or add at least two samples before counting.'
signal_id = sample_ids[0]  # Replace with your signal hypothesis.
background_ids = [sample_ids[1]]  # Replace with your background hypotheses.

channels = ['ee', 'mumu']
minimum_lepton_pt_gev = 20.0
maximum_abs_lepton_eta = 2.5
integrated_luminosity_fb = 1.0

signal = library[signal_id]
backgrounds = [library[sample_id] for sample_id in background_ids]
hypotheses = [signal, *backgrounds]
validate_analysis_context(hypotheses)

## 2. Explore distributions

This shape comparison applies the object preselection but no mass-window selection. It intentionally normalizes every source to unit area so any samples can be compared while developing the analysis.

In [ ]:
observable = 'mll'
display_range = (50.0, 130.0)
bins = 40

fig, ax = plt.subplots(figsize=(8, 5))
for index, sample in enumerate(hypotheses):
    frame = select_events(
        sample.load(),
        channels=channels,
        minimum_pt=minimum_lepton_pt_gev,
        maximum_abs_eta=maximum_abs_lepton_eta,
    )
    frame = frame.loc[frame[observable].between(*display_range)]
    weights = frame.get('weight', pd.Series(1.0, index=frame.index))
    weights = weights / weights.sum() if weights.sum() else weights
    role = 'Signal' if index == 0 else 'Background'
    ax.hist(frame[observable], bins=bins, range=display_range, weights=weights,
            histtype='step', linewidth=2, label=f'{role}: {sample.label}')
ax.set(xlabel='mll [GeV]', ylabel='Fraction of sample')
ax.grid(alpha=0.2)
ax.legend();

## 3. Fix the selection and calculate expected counts

For generated samples, expected counts are `1000 × cross_section_pb × luminosity_fb × efficiency`. Combine only samples with compatible configurations and normalization.

In [ ]:
mass_window_gev = (80.0, 100.0)
relative_background_uncertainty = 0.10

validate_counting_samples(hypotheses)
selected = {
    sample.sample_id: select_events(
        sample.expected_frame(integrated_luminosity_fb),
        channels=channels,
        minimum_pt=minimum_lepton_pt_gev,
        maximum_abs_eta=maximum_abs_lepton_eta,
        mass_window=mass_window_gev,
    )
    for sample in hypotheses
}
signal_frame = selected[signal_id]
background_frame = pd.concat(
    [selected[sample_id] for sample_id in background_ids], ignore_index=True
)
observed_frame = None  # Expected sensitivity only; no observed dataset selected.

In [ ]:
count = summarize_cut_and_count(
    signal_frame,
    background_frame,
    observed_frame,
    background_uncertainty_fraction=relative_background_uncertainty,
)
pd.Series(count.as_dict(), name='value').to_frame()

In [ ]:
pd.DataFrame([
    {
        'role': 'signal' if sample.sample_id == signal_id else 'background',
        'sample': sample.label,
        'selected_rows': len(selected[sample.sample_id]),
        'expected_yield': weighted_yield(selected[sample.sample_id]),
    }
    for sample in hypotheses
])

## 4. Define what an electron means (full-pipeline samples)

Delphes has already reconstructed electron *candidates*. The editable working point below decides which candidates count as analysis electrons. Changing these cuts does not rerun the detector simulation. Apply one definition to every signal and background in the comparison.

In [ ]:
full_pipeline_ids = [
    sample_id
    for sample_id, sample in library.items()
    if sample_id.startswith('run:') and sample.config.get('run_mode') == 'full'
]
pd.DataFrame([
    {
        'sample_id': sample_id,
        'label': library[sample_id].label,
        'collider_configuration': library[sample_id].analysis_context_label,
    }
    for sample_id in full_pipeline_ids
])

In [ ]:
# The default chooses the largest compatible group. Edit the result to choose roles.
compatible_groups = {}
for sample_id in full_pipeline_ids:
    context = library[sample_id].analysis_context
    compatible_groups.setdefault(context, []).append(sample_id)
object_sample_ids = max(compatible_groups.values(), key=len, default=[])[:3]
object_samples = [library[sample_id] for sample_id in object_sample_ids]
if object_samples:
    validate_analysis_context(object_samples)
else:
    print('Choose a full-pipeline sample to run the object exercise.')

### Editable electron and pairing definition

Try loose and tight isolation or acceptance requirements. The pairing rule matters when an event has more than two accepted electrons.

In [ ]:
electron_minimum_pt_gev = 20.0
electron_maximum_abs_eta = 2.5
electron_maximum_relative_isolation = 0.15
require_opposite_charge = True
pairing_rule = 'highest_pt_sum'  # or 'closest_to_z'

In [ ]:
def define_electrons_and_pairs(events):
    candidates = events.electrons
    accepted = candidates[
        (candidates.pt >= electron_minimum_pt_gev)
        & (abs(candidates.eta) <= electron_maximum_abs_eta)
        & (candidates.isolation <= electron_maximum_relative_isolation)
    ]
    pairs = ak.combinations(accepted, 2, fields=['first', 'second'])
    if require_opposite_charge:
        pairs = pairs[(pairs.first.charge * pairs.second.charge) < 0]
    pair_vectors = pairs.first + pairs.second
    if pairing_rule == 'highest_pt_sum':
        score = pairs.first.pt + pairs.second.pt
    elif pairing_rule == 'closest_to_z':
        score = -abs(pair_vectors.mass - 91.1876)
    else:
        raise ValueError('pairing_rule must be highest_pt_sum or closest_to_z')
    best_index = ak.argmax(score, axis=1, keepdims=True)
    best_pairs = ak.firsts(pairs[best_index])
    has_pair = ~ak.is_none(best_pairs)
    best_vectors = best_pairs[has_pair].first + best_pairs[has_pair].second
    return {
        'accepted_electrons': accepted,
        'has_pair': has_pair,
        'mll': ak.to_numpy(best_vectors.mass),
    }

In [ ]:
object_results = {}
for sample in object_samples:
    run_id = sample.sample_id.removeprefix('run:')
    events = open_run(run_id)
    result = define_electrons_and_pairs(events)
    object_results[sample.sample_id] = (events, result)

pd.DataFrame([
    {
        'sample': library[sample_id].label,
        'events': events.event_count,
        'events with >=1 accepted electron': int(
            ak.sum(ak.num(result['accepted_electrons']) >= 1)
        ),
        'events with selected pair': int(ak.sum(result['has_pair'])),
        'pair efficiency': float(ak.mean(result['has_pair'])),
    }
    for sample_id, (events, result) in object_results.items()
])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for sample_id, (_, result) in object_results.items():
    masses = result['mll']
    weights = np.ones(len(masses)) / len(masses) if len(masses) else None
    ax.hist(masses, bins=40, range=(50, 130), weights=weights,
            histtype='step', linewidth=2, label=library[sample_id].label)
ax.set(xlabel='Reconstructed dielectron mass [GeV]', ylabel='Fraction of sample')
ax.grid(alpha=0.2)
ax.legend();

Rerun the last three cells after changing the working point. Compare pair efficiency and the mass distribution for every hypothesis. Standard detector output is sufficient for these reconstructed-electron definitions; advanced output additionally exposes tracks, towers, particles, and particle-flow collections for deeper studies.